In [1]:
from pyspark.sql import SparkSession

In [13]:
import pyspark.sql.functions as F
from pyspark.sql import Window
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_1207/2270656995.py:11: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob4") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "12")\
    .config('spark.executor.memory', '7g')\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 12:21:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [18]:
spark.table('netscout_cdm.agg_subs_geo_home_work').count()

9916480

In [4]:
def haversine(long_x, lat_x, long_y, lat_y):
    from pyspark.sql import functions as F

    return F.acos(
        F.sin(F.toRadians(lat_x)) * F.sin(F.toRadians(lat_y)) +
        F.cos(F.toRadians(lat_x)) * F.cos(F.toRadians(lat_y)) *
            F.cos(F.toRadians(long_x) - F.toRadians(long_y))
    ) * F.lit(6371.0 * 1000)

In [5]:
def mock_getZidCentroidUDF(col_name):
    from pyspark.sql import functions as F
    return F.struct(
        (F.rand() * 10 + 54).cast("double").alias("lat"),   # ~54–64° (Россия)
        (F.rand() * 20 + 30).cast("double").alias("lon")    # ~30–50°
    )

In [21]:
def build_moves_in_home_work_locations(spark, logical_date, logger):
    from pyspark.sql import functions as F

    logger.info('Building DataFrame with retro data of home/work locations...')
    part_dt_0 = datetime(logical_date.year, logical_date.month, 22).date()
    
    for month_ago in range(4):
        cur_part_dt = part_dt_0 - relativedelta(months=month_ago)
        cur_home_work = (
            spark.table('netscout_cdm.agg_subs_geo_home_work')
            .withColumn(f'home_lat_{month_ago}', F.col('home_place_coord.lat'))
            .withColumn(f'home_lon_{month_ago}', F.col('home_place_coord.lon'))
            .withColumn(f'work_lat_{month_ago}', F.col('work_place_coord.lat'))
            .withColumn(f'work_lon_{month_ago}', F.col('work_place_coord.lon'))
            .select(
                'msisdn',
                f'home_lat_{month_ago}', f'home_lon_{month_ago}',
                f'work_lat_{month_ago}', f'work_lon_{month_ago}'
            )
        )
        if month_ago == 0:
            final_home_work = cur_home_work.selectExpr('*')
        else:
            final_home_work = (
                final_home_work
                .join(cur_home_work, ['msisdn'], 'left')
            )

    logger.info('Building features from retro data...')
    # make features from final data
    final_home_work_features = (
        final_home_work
        .withColumn('home_diff_0_1', haversine('home_lon_0', 'home_lat_0', 'home_lon_1', 'home_lat_1'))
        .withColumn('work_diff_0_1', haversine('work_lon_0', 'work_lat_0', 'work_lon_1', 'work_lat_1'))
        .withColumn('home_diff_0_2', haversine('home_lon_0', 'home_lat_0', 'home_lon_2', 'home_lat_2'))
        .withColumn('work_diff_0_2', haversine('work_lon_0', 'work_lat_0', 'work_lon_2', 'work_lat_2'))
        .withColumn('home_diff_0_3', haversine('home_lon_0', 'home_lat_0', 'home_lon_3', 'home_lat_3'))
        .withColumn('work_diff_0_3', haversine('work_lon_0', 'work_lat_0', 'work_lon_3', 'work_lat_3'))

        .withColumn('home_diff_1_2', haversine('home_lon_1', 'home_lat_1', 'home_lon_2', 'home_lat_2'))
        .withColumn('work_diff_1_2', haversine('work_lon_1', 'work_lat_1', 'work_lon_2', 'work_lat_2'))
        .withColumn('home_diff_1_3', haversine('home_lon_1', 'home_lat_1', 'home_lon_3', 'home_lat_3'))
        .withColumn('work_diff_1_3', haversine('work_lon_1', 'work_lat_1', 'work_lon_3', 'work_lat_3'))

        .withColumn('home_diff_2_3', haversine('home_lon_2', 'home_lat_2', 'home_lon_3', 'home_lat_3'))
        .withColumn('work_diff_2_3', haversine('work_lon_2', 'work_lat_2', 'work_lon_3', 'work_lat_3'))
        
        .select(
            'msisdn',
            'home_diff_0_1', 'work_diff_0_1',
            'home_diff_0_2', 'work_diff_0_2',
            'home_diff_0_3', 'work_diff_0_3',
            'home_diff_1_2', 'work_diff_1_2',
            'home_diff_1_3', 'work_diff_1_3',
            'home_diff_2_3', 'work_diff_2_3'
        )

        .limit(8_000_000)

        .withColumn('_rn', F.row_number().over(Window.partitionBy('msisdn').orderBy(F.lit(1))))
        .filter(F.col('_rn') == 1)
        .drop('_rn')
    )

    return final_home_work_features

In [17]:
log = logging.getLogger(__name__)
logical_date = datetime.now().date()

In [12]:
build_moves_in_home_work_locations(spark, logical_date, log)

/opt/spark/python/pyspark/sql/functions.py:1662: FutureWarning: Deprecated in 2.1, use radians instead.
  warnings.warn("Deprecated in 2.1, use radians instead.", FutureWarning)


DataFrame[msisdn: int, home_diff_0_1: double, work_diff_0_1: double, home_diff_0_2: double, work_diff_0_2: double, home_diff_0_3: double, work_diff_0_3: double, home_diff_1_2: double, work_diff_1_2: double, home_diff_1_3: double, work_diff_1_3: double, home_diff_2_3: double, work_diff_2_3: double]

In [13]:
build_moves_in_home_work_locations(spark, logical_date, log).count()

2000000

In [22]:
build_moves_in_home_work_locations(spark, logical_date, log).count()

500000

In [24]:
df = build_moves_in_home_work_locations(spark, logical_date, log)

In [25]:
(
    df
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.mob_features_dm__4__base")
)

26/03/14 12:31:59 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/14 12:32:48 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.7: Command exited with code 137
26/03/14 12:32:49 WARN TaskSetManager: Lost task 12.0 in stage 40.0 (TID 227) (172.18.0.7 executor 1): ExecutorLostFailure (executor 1 exited caused by one of the running tasks) Reason: Command exited with code 137
26/03/14 12:32:49 WARN TaskSetManager: Lost task 13.0 in stage 40.0 (TID 228) (172.18.0.7 executor 1): ExecutorLostFailure (executor 1 exited caused by one of the running tasks) Reason: Command exited with code 137
26/03/14 12:32:52 WARN TaskSetManager: Lost task 1.1 in stage 40.0 (TID 236) (172.18.0.7 executor 2): FetchFailed(null, shuffleId=12, mapIndex=-1, mapId=-1, reduceId=15, message=
org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 12 partition 1